# E-LOWFREQ — does device identity survive in a *different frequency band*?

**Measurement only. No training, no generation. ~2 h.**

*Beyond PRNU* (arXiv 2111.02144; PMC9609198) reports a device-specific signal in **low- and
mid-frequency DCT bands** that is more location-independent and more robust to processing than
classical high-frequency PRNU. Berdich et al. (*Entropy* 2022, 24, 1158) identify smartphones from
low-mid frequency DCT coefficients of dark frames. Both are cited *against* any unrestricted
"camera identity" claim.

Right now the paper **concedes** that limitation. This experiment **tests** it.

### The construction
For each device, from the same E1/E2 splits and the same images:
1. block DCT (8×8) of the luminance crop;
2. retain a **low/mid band** — coefficients with `LO ≤ i+j ≤ HI`, excluding DC;
3. per-image mean-removed band map, averaged over the split → device signature **L_d**;
4. row/column zero-mean and the same Wiener-in-DFT cleaning used for PRNU.

Measurement is the **identical paired statistic**: θ = ρ(gens → L_own) − ρ(gens → L_other), over
the **existing** generations. Nothing else in the pipeline changes.

### Both outcomes are publishable
| result | consequence |
|---|---|
| **L survives generation while PRNU does not** | A **positive** result. Provenance survives in a different band; a substantially bigger paper, and it inverts the headline |
| **L also fails** | The claim broadens from "PRNU-derived" to **two independent device-identity representations**, and the *Beyond PRNU* constraint is retired by testing rather than conceding |

**Gate first (L0/L1).** If L cannot separate the devices on *real* images, nothing downstream means
anything and the experiment stops.

In [ ]:
# %% L-CFG
import os, sys, json, glob, math, time, gc, hashlib, subprocess, itertools
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
import csv as _csv
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE="/content/drive/MyDrive"
except Exception:
    DRIVE=os.environ.get("EINV_DRIVE",".")
BASE=os.path.join(DRIVE,"inv_channel/E_INV_P0_v3")
SEED=os.path.join(DRIVE,"inv_channel/E_SEEDEXT")
ROOT=os.path.join(DRIVE,"inv_channel/E_LOWFREQ")
for d in ["","fingerprints","csv","figs"]: os.makedirs(os.path.join(ROOT,d),exist_ok=True)

@dataclass
class CFG:
    STAGES: tuple = ("L0","L1","L2","L3")
    CROP: int = 1024
    BLK: int = 8                    # DCT block size
    LO: int = 1                     # keep coefficients with LO <= i+j <= HI (DC excluded)
    HI: int = 5                     # 8x8 block: i+j runs 0..14; 1..5 is low/mid
    ROLES: tuple = ("A","B","C","D","D2")
    PAIR: tuple = ("A","B")         # the same-model pair
    G_PER: int = 500
    N_PERM: int = 10000
    ALPHA: float = 0.01
C=CFG(); MEAS=C.CROP
subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","scipy"],check=False)
import numpy as np, pandas as pd, torch
import torch.nn.functional as tF
from PIL import Image
from scipy.fft import dctn, idctn
from scipy import stats as sps
DEV="cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise"); _MD=DEV
Image.MAX_IMAGE_PIXELS=None
def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False
def append_row(p,h,r):
    new=not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p,cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o,open(t,"w"),**kw); os.replace(t,p)

MAN={}
for r in _csv.DictReader(open(os.path.join(BASE,"manifest","manifest.csv"))):
    MAN.setdefault((r["role"],r["split"]),[]).append(r["path"])
print(f"manifest loaded | band: {C.LO} <= i+j <= {C.HI} of an {C.BLK}x{C.BLK} DCT block "
      f"(DC excluded) | device={DEV}")

In [ ]:
# %% low/mid-band extractor + the SAME cleaning used for PRNU
def load_lum(fp):
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        if min(W,H)<MEAS: HALT(f"{fp}: {W}x{H} < {MEAS}")
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    return (0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)

_MASK=None
def band_mask():
    """8x8 mask keeping LO <= i+j <= HI, DC (0,0) excluded."""
    global _MASK
    if _MASK is None:
        m=np.zeros((C.BLK,C.BLK),np.float32)
        for i in range(C.BLK):
            for j in range(C.BLK):
                if C.LO<=i+j<=C.HI and not (i==0 and j==0): m[i,j]=1.0
        _MASK=m
        print(f"[band] keeping {int(m.sum())} of {C.BLK*C.BLK} coefficients per block")
    return _MASK

def lowmid_map(Y):
    """Block DCT -> keep the low/mid band -> inverse -> a spatial map of band energy deviation."""
    n=MEAS//C.BLK
    B=Y.reshape(n,C.BLK,n,C.BLK).transpose(0,2,1,3)              # (n,n,8,8) blocks
    D=dctn(B,axes=(2,3),norm="ortho")
    D=D*band_mask()[None,None]
    R=idctn(D,axes=(2,3),norm="ortho")
    out=R.transpose(0,2,1,3).reshape(MEAS,MEAS).astype(np.float32)
    out=out-out.mean(); sd=float(out.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate low/mid map")
    return (out/sd).astype(np.float32)

def _c2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MD=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
def zm(K):
    K=K-K.mean(axis=1,keepdims=True); return K-K.mean(axis=0,keepdims=True)
def wdft(K):
    F=np.fft.fft2(K.astype(np.float32)); mag=(np.abs(F)/K.shape[0]).astype(np.float32)
    sig=max(float(np.median(mag))/0.6745,1e-12); s2=np.float32(sig**2); vm=None
    for w in (3,5,7,9):
        k=np.ones((w,w),np.float32)/(w*w); mu=_c2(mag,k)
        v=np.maximum(_c2(mag*mag,k)-mu*mu,0.0); vm=v if vm is None else np.minimum(vm,v)
    return np.real(np.fft.ifft2(F*(s2/(vm+s2)).astype(np.float32))).astype(np.float32)

def estimate_L(files):
    """Device signature: mean low/mid map over a split, then the SAME cleaning as PRNU."""
    acc=None
    for fp in files:
        m=lowmid_map(load_lum(fp))
        acc=m if acc is None else acc+m
    return wdft(zm((acc/len(files)).astype(np.float32)))

def ncc(a,b):
    a=a-a.mean(); b=b-b.mean()
    na=float(np.linalg.norm(a)); nb=float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))
def nccb(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)

In [ ]:
# %% L0 — build L_d per device, and CHECK IT SEPARATES DEVICES ON REAL IMAGES
#     This is the gate. If L cannot do source-camera ID on real photographs, stop here.
def l0():
    for role in C.ROLES:
        for which in ("E1","E2"):
            p=os.path.join(ROOT,"fingerprints",f"L_{role}_{which}.npy")
            if os.path.exists(p): continue
            files=MAN.get((role,which))
            if not files: continue
            t0=time.time(); np.save(p,estimate_L(files))
            print(f"[L0] L_{role}_{which} from {len(files)} imgs ({time.time()-t0:.0f}s)")
    L2={r:np.load(os.path.join(ROOT,"fingerprints",f"L_{r}_E2.npy"))
        for r in C.ROLES if os.path.exists(os.path.join(ROOT,"fingerprints",f"L_{r}_E2.npy"))}
    A,B=C.PAIR
    print(f"\n[L0] NCC(L_A, L_B) same-model = {ncc(L2[A],L2[B]):+.4f}   "
          f"(PRNU kappa_model was 0.0073)")
    print(f"[L0] NCC(L_A_E1, L_A_E2) split-half = "
          f"{ncc(np.load(os.path.join(ROOT,'fingerprints',f'L_{A}_E1.npy')),L2[A]):+.4f}   "
          f"(PRNU was 0.362)")

    # positive control on held-out REAL images — the gate
    p=os.path.join(ROOT,"csv","l0_positive_control.csv")
    if not os.path.exists(p):
        for role in (A,B):
            for i,fp in enumerate(MAN[(role,"H")]):
                Y=load_lum(fp); Wm=lowmid_map(Y)
                for k_ in L2:
                    append_row(p,["role_img","role_L","idx","rho"],
                               [role,k_,i,f"{ncc(Wm,Y*L2[k_]):.6e}"])
            print(f"[L0] positive control {role} done")
    d=pd.read_csv(p)
    same=d[d.role_img==d.role_L]["rho"].to_numpy()
    cross=d[(d.role_img!=d.role_L)&(d.role_L.isin(C.PAIR))]["rho"].to_numpy()
    lab=np.r_[np.ones(len(same)),np.zeros(len(cross))]; sc=np.r_[same,cross]
    o=np.argsort(sc); rk=np.empty(len(sc)); rk[o]=np.arange(len(sc))
    auc=float((rk[lab==1].mean()-(lab.sum()-1)/2)/(len(sc)-lab.sum()))
    # the paired real-image contrast — the DENOMINATOR, same statistic as PRNU's R_real
    def rc(role,other):
        a=d[(d.role_img==role)&(d.role_L==role)].sort_values("idx")["rho"].to_numpy()
        b=d[(d.role_img==role)&(d.role_L==other)].sort_values("idx")["rho"].to_numpy()
        n=min(len(a),len(b)); return a[:n]-b[:n]
    RA,RB=rc(A,B),rc(B,A); R_real=float(0.5*(RA.mean()+RB.mean()))
    se=float(np.sqrt((RA.var(ddof=1)/len(RA)+RB.var(ddof=1)/len(RB))/4))
    print(f"\n[L0] same-model AUC on REAL images = {auc:.4f}   (PRNU: 1.0000)")
    print(f"[L0] R_real(lowmid) = {R_real:.5e}  SE {se:.3e}  lower99 "
          f"{R_real-sps.norm.ppf(0.99)*se:.5e}   (PRNU R_real: 3.567e-02)")
    ajson({"auc":auc,"R_real":R_real,"se":se,
           "R_real_lower99":float(R_real-sps.norm.ppf(0.99)*se),
           "ncc_AB":ncc(L2[A],L2[B]),"band":[C.LO,C.HI,C.BLK]},
          os.path.join(ROOT,"fingerprints","l0_gates.json"),indent=2)
    print("\n" + "="*70)
    if auc < 0.75:
        print("GATE FAILED: the low/mid band does not separate these devices on real images.")
        print("STOP. Either widen the band (raise HI), or report that classical PRNU is the only")
        print("device representation this dataset supports at 1024px centre crop.")
    else:
        print(f"GATE PASSED (AUC {auc:.4f}). The low/mid band carries device identity on real")
        print("images. Proceed to L1 (VAE) and L2 (generations).")
    print("="*70)
if stage("L0"): l0()

In [ ]:
# %% L1 — does the low/mid signature survive the AUTOENCODER? (same test as S1b)
def l1():
    from diffusers import AutoencoderKL
    A,B=C.PAIR
    L2={r:np.load(os.path.join(ROOT,"fingerprints",f"L_{r}_E2.npy")) for r in C.PAIR}
    vae=AutoencoderKL.from_pretrained("stabilityai/stable-diffusion-3.5-medium",subfolder="vae",
                                      torch_dtype=torch.float32,token=True).to(DEV).eval()
    p=os.path.join(ROOT,"csv","l1_vae.csv")
    if not os.path.exists(p):
        for role in C.PAIR:
            for i,fp in enumerate(MAN[(role,"H")]):
                with Image.open(fp) as im:
                    im=im.convert("RGB"); W,H=im.size
                    rgb=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,
                                            (H-MEAS)//2+MEAS)),np.float32)
                with torch.no_grad():
                    x=torch.from_numpy(rgb/127.5-1.0).permute(2,0,1)[None].to(DEV)
                    rec=vae.decode(vae.encode(x).latent_dist.mode()).sample[0]
                    rec=rec.permute(1,2,0).cpu().numpy()
                R=np.clip((rec+1.0)*127.5,0,255).astype(np.float32)
                for stage_,img in (("pre",rgb),("post",R)):
                    Y=(0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]).astype(np.float32)
                    Wm=lowmid_map(Y)
                    append_row(p,["role","stage","idx","rho_A","rho_B"],
                               [role,stage_,i,f"{ncc(Wm,Y*L2[A]):.6e}",f"{ncc(Wm,Y*L2[B]):.6e}"])
            print(f"[L1] {role} done")
    del vae; gc.collect(); torch.cuda.empty_cache()
    d=pd.read_csv(p)
    def contr(role,stage_):
        g=d[(d.role==role)&(d.stage==stage_)].sort_values("idx")
        own,oth=("rho_A","rho_B") if role==A else ("rho_B","rho_A")
        return g[own].to_numpy()-g[oth].to_numpy()
    pre=np.r_[contr(A,"pre"),contr(B,"pre")]; post=np.r_[contr(A,"post"),contr(B,"post")]
    eta=float(post.mean()/pre.mean())
    print(f"\n[L1] R_lowmid pre {pre.mean():.5e}  post {post.mean():.5e}")
    print(f"[L1] device-specific retention eta(lowmid) = {eta:.4f}   (PRNU eta was 0.3661)")
    ajson({"pre":float(pre.mean()),"post":float(post.mean()),"eta":eta},
          os.path.join(ROOT,"fingerprints","l1_vae.json"),indent=2)
if stage("L1"): l1()

In [ ]:
# %% L2 — measure the EXISTING generations against the low/mid signatures
def gdir(tag,root): return os.path.join(root,"gens",tag)
TAGS=[(f"{a}_raw_s{s}_r16", BASE if s<3 else SEED) for a in C.PAIR for s in range(6)]
def l2():
    L2={r:torch.from_numpy(np.load(os.path.join(ROOT,"fingerprints",f"L_{r}_E2.npy"))).to(_MD)
        for r in C.PAIR}
    K=torch.stack([L2[r] for r in C.PAIR])
    p=os.path.join(ROOT,"csv","l2_measure.csv"); done=done_keys(p,["tag","gen_idx","L"])
    ex=ThreadPoolExecutor(max_workers=4)
    try:
        for tag,root in TAGS:
            paths=[os.path.join(gdir(tag,root),f"{i:05d}.png") for i in range(C.G_PER)]
            todo=[i for i in range(C.G_PER) if not all((tag,str(i),r) in done for r in C.PAIR)]
            if not todo: continue
            fut={i:ex.submit(load_lum,paths[i]) for i in todo[:6]}; t0=time.time()
            for pos,i in enumerate(todo):
                Y=fut.pop(i).result(); nx=pos+6
                if nx<len(todo): fut[todo[nx]]=ex.submit(load_lum,paths[todo[nx]])
                Wm=torch.from_numpy(lowmid_map(Y)).to(_MD); Z=torch.from_numpy(Y).to(_MD)
                v=nccb(Wm,Z.unsqueeze(0)*K).cpu().numpy()
                for j,r in enumerate(C.PAIR):
                    if (tag,str(i),r) in done: continue
                    append_row(p,["tag","gen_idx","L","rho"],[tag,i,r,f"{v[j]:.6e}"])
            print(f"[L2] {tag}: {len(todo)} measured ({time.time()-t0:.0f}s)")
    finally:
        ex.shutdown(wait=True); gc.collect()
        if DEV=="cuda": torch.cuda.empty_cache()
    n=len(done_keys(p,["tag","gen_idx","L"])); exp=len(TAGS)*C.G_PER*len(C.PAIR)
    print(f"[L2] {n}/{exp} rows"); 
    if n!=exp: HALT("L2 incomplete")
if stage("L2"): l2()

In [ ]:
# %% L3 — the same IUT and the same bound, in the low/mid band
def l3():
    rng=np.random.default_rng(0)
    A,B=C.PAIR
    d=pd.read_csv(os.path.join(ROOT,"csv","l2_measure.csv"))
    g0=json.load(open(os.path.join(ROOT,"fingerprints","l0_gates.json")))
    R_low=g0["R_real_lower99"]
    def th(tag,own,oth):
        a=d[(d.tag==tag)&(d.L==own)].sort_values("gen_idx")["rho"].to_numpy(float)
        b=d[(d.tag==tag)&(d.L==oth)].sort_values("gen_idx")["rho"].to_numpy(float)
        n=min(len(a),len(b)); return a[:n]-b[:n]
    def sf(x,n=C.N_PERM):
        o=x.mean(); c=sum((x*rng.choice((-1.,1.),len(x))).mean()>=o for _ in range(n))
        return (c+1)/(n+1)
    out={"band":[C.LO,C.HI,C.BLK],"R_real_lower99":R_low,"arms":{}}
    Us=[]
    print(f"{'arm':4s} {'adapter':18s} {'theta':>12s}")
    for arm,own,oth in ((A,A,B),(B,B,A)):
        v=[]
        for tag,_ in [t for t in TAGS if t[0].startswith(arm)]:
            x=th(tag,own,oth)
            if len(x): v.append(x); print(f"{arm:4s} {tag:18s} {x.mean():+12.3e}")
        m=np.array([x.mean() for x in v]); k=len(m)
        tc=float(sps.t.ppf(1-C.ALPHA/2,df=k-1)); U=float(m.mean()+tc*m.std(ddof=1)/np.sqrt(k))
        Us.append(U)
        out["arms"][arm]={"n":k,"means":m.tolist(),"theta":float(m.mean()),
                          "sd":float(m.std(ddof=1)),"U":U,
                          "p_adapter":sf(m),"p_image":sf(np.concatenate(v))}
        print(f"     -> n={k} theta={m.mean():+.3e} sd={m.std(ddof=1):.3e} U={U:+.3e} "
              f"p_adapter={out['arms'][arm]['p_adapter']:.4g}\n")
    IUT=all(out["arms"][a]["p_adapter"]<C.ALPHA for a in C.PAIR)
    U=max(Us); out["U_device"]=U; out["lambda_U"]=U/R_low; out["IUT_pass"]=bool(IUT)
    l1p=os.path.join(ROOT,"fingerprints","l1_vae.json")
    if os.path.exists(l1p): out["eta_vae"]=json.load(open(l1p))["eta"]
    print("="*72)
    print(f"LOW/MID BAND — IUT: {'PASS — DEVICE IDENTITY TRANSFERS IN THIS BAND' if IUT else 'FAIL — the null holds here too'}")
    print(f"  lambda_U(lowmid) = {U/R_low:.4%}      (PRNU lambda_U = 0.32%)")
    if "eta_vae" in out:
        print(f"  eta(lowmid) = {out['eta_vae']:.4f}   (PRNU eta = 0.3661)")
    print("="*72)
    print("READ:")
    print("  IUT PASS  -> a POSITIVE result. Provenance survives in a band classical PRNU misses;")
    print("               this inverts the headline and is a substantially bigger paper.")
    print("  IUT FAIL  -> the claim broadens from 'PRNU-derived' to TWO INDEPENDENT device-identity")
    print("               representations, and the Beyond PRNU constraint is retired by TEST.")
    ajson(out,os.path.join(ROOT,"E_LOWFREQ_results.json"),indent=2)
if stage("L3"): l3()

## Running it

| stage | wall | check |
|---|---|---|
| L0 | ~25 min | **the gate** — same-model AUC on real images. Below 0.75, stop |
| L1 | ~10 min | η(lowmid) vs PRNU's 0.3661 |
| L2 | ~60 min | 6000 generations × 2 signatures |
| L3 | ~1 min | the verdict |

**If L0 fails**, the honest report is that at a 1024 centre crop this dataset supports only the
classical PRNU representation — try `HI = 7` or `BLK = 16` once, state that you did, and stop.
Do not tune the band until something passes.

**Pre-register before running:** the band is `1 ≤ i+j ≤ 5` of an 8×8 DCT with DC excluded; the
gate is same-model AUC ≥ 0.75 on held-out real images; the primary test is the same
adapter-clustered IUT at α = 0.01 over the same six seeds per arm. Changing any of these after
seeing L3 is the one thing that would undo this study's pre-registration credibility.